In [39]:
import import_ipynb
import numpy as np
from Needleman_Wunsch_Algorithm import *

In [40]:
seq1 = "TATCCAGTCATTTC"
seq2 = "ATGCCAAGTCCTT"

In [41]:
def initiate_smith_matrix(seq1,seq2):
    zero = 0
    row_ls=generate_col_row_ls(seq1, zero)
    col_ls =generate_col_row_ls(seq2,zero)
    mat = np.zeros((len(col_ls), len(row_ls)), dtype=int)
    mat[0,:] = row_ls
    mat[:,0] = col_ls
    return mat


In [42]:
mat = initiate_smith_matrix(seq1,seq2)


In [43]:
def fill_smith_matrix(mat,gap,match,mismatch,seq1,seq2):
    
    pointer = np.zeros((len(mat), len(mat[0])), dtype=object)

    for i in range(1, len(mat)):
        pointer[i,0] = None  
    for j in range(1, len(mat[0])):
        pointer[0,j] = None  

        
    for i in range(1, len(mat)):
        for j in range(1,len(mat[0])):            
            across_val = across(mat,gap,i,j)
            vertical_val = vertical(mat,gap,i,j)
            diagonal_val = diagonal(mat,i,j,match,mismatch,seq1,seq2)
            mat[i,j] = max(0,across_val, vertical_val, diagonal_val)

            if mat[i,j] == 0:
                pointer[i,j] = "STOP"
            #If diagonal is max; a character from each sequence was aligned
            elif mat[i,j] == diagonal_val:
                pointer[i,j] = "D"
            #If vertical was max; a gap was introduced in sequence 1
            elif mat[i,j] == vertical_val:
                pointer[i,j] = "V"
            #if across was max; a gap was introduced in sequence 2 
            else:
                pointer[i,j] = "A"
            
    return mat, pointer


mat, point = fill_smith_matrix(mat,-2,1,-1,seq1,seq2)
print(point)

[[0 None None None None None None None None None None None None None None]
 [None 'STOP' 'D' 'STOP' 'STOP' 'STOP' 'D' 'STOP' 'STOP' 'STOP' 'D'
  'STOP' 'STOP' 'STOP' 'STOP']
 [None 'D' 'STOP' 'D' 'STOP' 'STOP' 'STOP' 'STOP' 'D' 'STOP' 'STOP' 'D'
  'D' 'D' 'STOP']
 [None 'STOP' 'STOP' 'STOP' 'D' 'STOP' 'STOP' 'D' 'STOP' 'STOP' 'STOP'
  'STOP' 'D' 'STOP' 'STOP']
 [None 'STOP' 'STOP' 'STOP' 'D' 'D' 'STOP' 'STOP' 'STOP' 'D' 'STOP'
  'STOP' 'STOP' 'STOP' 'D']
 [None 'STOP' 'STOP' 'STOP' 'D' 'D' 'D' 'STOP' 'STOP' 'D' 'STOP' 'STOP'
  'STOP' 'STOP' 'D']
 [None 'STOP' 'D' 'STOP' 'STOP' 'STOP' 'D' 'A' 'STOP' 'STOP' 'D' 'STOP'
  'STOP' 'STOP' 'STOP']
 [None 'STOP' 'D' 'STOP' 'STOP' 'STOP' 'D' 'D' 'STOP' 'STOP' 'D' 'D'
  'STOP' 'STOP' 'STOP']
 [None 'STOP' 'STOP' 'STOP' 'STOP' 'STOP' 'STOP' 'D' 'D' 'STOP' 'STOP'
  'STOP' 'STOP' 'STOP' 'STOP']
 [None 'D' 'STOP' 'D' 'STOP' 'STOP' 'STOP' 'STOP' 'D' 'A' 'STOP' 'D' 'D'
  'D' 'STOP']
 [None 'STOP' 'STOP' 'STOP' 'D' 'D' 'STOP' 'STOP' 'V' 'D' 'A' 'STOP'
 

In [44]:
def traceback_smith(mat, seq1, seq2, pointer):
    
    i, j = np.unravel_index(mat.argmax(), mat.shape)
    print(mat[i][j])

    
    seq1_ls = ["-"]
    seq2_ls = ["-"]

    #for each nucleotide in sequence 1; add to sequence 1 list
    for nuc in seq1:          
        seq1_ls.append(nuc)

    #for each nucleotide in sequence 2; add to sequence 1 list
    for nuc in seq2:          
        seq2_ls.append(nuc)

    #Make sequence alignment strings   
    seq1_alignment = ""
    seq2_alignment = ""

    while pointer[i,j] is not None and pointer[i,j] != "Stop": #while the matrix is not None and and equal to STOP
        if pointer[i,j] == "D": #if the pointer is D; came diagonally
            seq1_alignment = seq1_ls[j] + seq1_alignment #the current nucleotide gets added to the list
            seq2_alignment = seq2_ls[i] + seq2_alignment #the current nucleotide gets added to the list
            
            #Move diagonally across 
            i -= 1 
            j -= 1

        elif pointer[i, j] == "V":
            seq1_alignment = seq1_ls[i] + seq1_alignment  # Current nucleotide gets added to the list
            seq2_alignment = "-" + seq2_alignment         #Gap added to sequence 2

            #move up one row
            i -= 1

        elif pointer[i, j] == "A":
            seq1_alignment = "-" + seq1_alignment        #Gap added to sequence 1
            seq2_alignment = seq2_ls[j] + seq2_alignment #Current nucleotide gets added to the list
            
            #move back one column
            j -= 1

        #If it is equal to 0 then the loop will break 
        else:
            break
    return seq1_alignment, seq2_alignment

In [45]:
traceback_smith(mat,seq1,seq2,point)

5


('AGTCATT', 'AGTCCTT')

In [46]:
def smith_waterman_alignment(seq1,seq2,gap,match,mismatch):
    mat =initiate_smith_matrix(seq1,seq2)
    mat, pointer = fill_smith_matrix(mat,gap,match,mismatch,seq1,seq2)
    print(mat)
    seq1_alignment, seq2_alignment = traceback_smith(mat, seq1, seq2, pointer)
    print(f"Sequence 1: {seq1_alignment}")
    print(f"Sequence 2: {seq2_alignment}")
    return seq1_alignment, seq2_alignment

https://stackoverflow.com/questions/55203488/how-do-i-find-the-index-of-the-greatest-number-of-a-matrix-in-python (how to find the index of maximum value in a matrix)

In [47]:
smith_waterman_alignment(seq1,seq2,-2,1,-1)

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 1 0 0 0 1 0 0 0 0]
 [0 1 0 2 0 0 0 0 1 0 0 2 1 1 0]
 [0 0 0 0 1 0 0 1 0 0 0 0 1 0 0]
 [0 0 0 0 1 2 0 0 0 1 0 0 0 0 1]
 [0 0 0 0 1 2 1 0 0 1 0 0 0 0 1]
 [0 0 1 0 0 0 3 1 0 0 2 0 0 0 0]
 [0 0 1 0 0 0 1 2 0 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 2 1 0 0 0 0 0 0]
 [0 1 0 1 0 0 0 0 3 1 0 1 1 1 0]
 [0 0 0 0 2 1 0 0 1 4 2 0 0 0 2]
 [0 0 0 0 1 3 1 0 0 2 3 1 0 0 1]
 [0 1 0 1 0 1 2 0 1 0 1 4 2 1 0]
 [0 1 0 1 0 0 0 1 1 0 0 2 5 3 1]]
5
Sequence 1: AGTCATT
Sequence 2: AGTCCTT


('AGTCATT', 'AGTCCTT')